# Drishti — OCR Spike (standalone, CPU-only)

Split out of `00_feasibility_spike_colab.ipynb` on purpose. Running PaddleOCR in a process
that already has the VLMs loaded **hard-kills the kernel**
(`AsyncIOLoopKernelRestarter: restarting kernel`) — the process dies, so you get no Python
traceback. Observed on Colab; the same collision applies anywhere. Two causes stack:

- **PyTorch and PaddlePaddle each bundle their own OpenMP runtime.** Co-loading both in one
  process is a well-known segfault source.
- **Memory.** The two VLMs are ~8.5 GB of weights, and Colab's free tier has ~12.7 GB of
  system RAM — little headroom once PaddlePaddle and its models load too.

So this notebook is deliberately minimal:
- **no torch, no transformers, no VLMs** — no OpenMP clash, no memory pressure
- CPU-only PaddleOCR (PP-OCR models are small)
- runs in a **fresh Colab runtime** or **locally in VS Code** — it needs no GPU

Verifies two load-bearing assumptions for Drishti:
1. Medicine mode — can OCR read a real strip's **drug name, EXP, MRP**? (`lang='en'`)
2. Read mode — can it read **Devanagari** (Marathi/Hindi)? (`lang='devanagari'`)

> **Run this in a fresh runtime** — not the one where §2a/§2b already loaded the VLMs.
> That collision is the whole reason this notebook is separate.
>
> Keep the VLM cells (`00_..._colab.ipynb` §2a/§2b) and the VizWiz baseline
> (`01_vizwiz_baseline.ipynb`) on Colab with a T4 GPU — those genuinely need the GPU.

In [1]:
# CPU build: matches the eventual phone target, and CPU timings are the honest number
# for an on-device app. No GPU/CUDA needed anywhere in this notebook.
%pip install -q paddlepaddle paddleocr

import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
from PIL import Image

print('paddleocr version:', version('paddleocr'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/

## 1. Load your medicine-strip photo

**Colab:** leave `IMAGE_PATHS` empty — an upload widget appears.
**Local (VS Code):** set `IMAGE_PATHS` to your photo path(s).

In [2]:
# Point these at real photos: back of a medicine strip (name + EXP + MRP visible).
# Leave empty on Colab to get an upload widget instead.
IMAGE_PATHS = [
    # r'C:\Users\devgu\Downloads\strip1.jpg',
]

images, labels = [], []

if IMAGE_PATHS:
    for p in IMAGE_PATHS:
        path = Path(p)
        if not path.exists():
            raise FileNotFoundError(f'not found: {path}')
        images.append(Image.open(path).convert('RGB'))
        labels.append(path.name)
else:
    try:
        from google.colab import files
        for name in files.upload():
            images.append(Image.open(name).convert('RGB'))
            labels.append(name)
    except ImportError:
        raise SystemExit('Not on Colab — set IMAGE_PATHS above to your photo(s).')

print(f'{len(images)} image(s) loaded:', ', '.join(labels))
for img, name in zip(images, labels):
    print(f'  {name}: {img.size[0]}x{img.size[1]}')

Saving IMG_20260801_203248259.jpg to IMG_20260801_203248259.jpg
Saving IMG_20260801_203256207.jpg to IMG_20260801_203256207.jpg
Saving IMG_20260801_203301122.jpg to IMG_20260801_203301122.jpg
3 image(s) loaded: IMG_20260801_203248259.jpg, IMG_20260801_203256207.jpg, IMG_20260801_203301122.jpg
  IMG_20260801_203248259.jpg: 4080x3072
  IMG_20260801_203256207.jpg: 4080x3072
  IMG_20260801_203301122.jpg: 4080x3072


## 2. Run OCR — English and Devanagari

In [3]:
from paddleocr import PaddleOCR


def extract_lines(result):
    """Normalize PaddleOCR output to [(confidence, text)].

    Handles both result shapes so this survives a version bump:
      3.x .predict() -> objects/dicts carrying `rec_texts` + `rec_scores`
      2.x .ocr()     -> nested [[bbox, (text, score)], ...]
    """
    lines = []
    for page in result or []:
        texts = scores = None

        if isinstance(page, dict):
            texts, scores = page.get('rec_texts'), page.get('rec_scores')
        else:
            texts = getattr(page, 'rec_texts', None)
            scores = getattr(page, 'rec_scores', None)
            if texts is None and hasattr(page, 'json'):
                blob = page.json
                blob = blob.get('res', blob) if isinstance(blob, dict) else {}
                texts, scores = blob.get('rec_texts'), blob.get('rec_scores')

        if texts is not None:
            scores = scores if scores is not None else [float('nan')] * len(texts)
            lines.extend(zip(scores, texts))
            continue

        if isinstance(page, list):          # 2.x layout
            for item in page:
                try:
                    _bbox, (text, score) = item
                    lines.append((score, text))
                except (TypeError, ValueError):
                    continue
    return lines


def run_paddle(imgs, lang):
    """Run OCR, preferring the 3.x predict() API and falling back to 2.x ocr()."""
    ocr = PaddleOCR(lang=lang)
    out, t0 = [], time.time()
    for img in imgs:
        arr = np.array(img)
        out.extend(extract_lines(ocr.predict(arr) if hasattr(ocr, 'predict') else ocr.ocr(arr)))
    return out, time.time() - t0


results = {}
for lang in ('en', 'devanagari'):
    try:
        lines, secs = run_paddle(images, lang)
        results[lang] = lines
        print(f'\n=== lang="{lang}" — {secs:.1f}s (CPU), {len(lines)} lines ===')
        for conf, text in lines:
            print(f'  {conf:.2f}  {text}')
    except Exception as e:
        results[lang] = []
        print(f'\n=== lang="{lang}" FAILED: {type(e).__name__}: {e} ===')

# medicine mode reads Latin-script fields (drug name / EXP / MRP)
ocr_lines = results.get('en', [])
ocr_text = ' '.join(text for _, text in ocr_lines)

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/UVDoc`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Creating model: ('PP-OCRv6_medium_rec', None, None)
Using official model (PP-OCRv6_medium_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Resized image size (4080x3072) exceeds max_side_limit of 4000. Resizing to fit within limit.



=== lang="en" FAILED: NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at /paddle/paddle/fluid/framework/new_executor/instruction/onednn/onednn_instruction.cc:116)
 ===

=== lang="devanagari" FAILED: ValueError: No models are available for lang='devanagari' and ocr_version=None. ===


## 3. Expiry / MRP extraction

Same patterns as `app/parsers.py` — keep them in sync if you tune them here.

In [4]:
import re

date_pat = re.compile(r'(?:EXP|Expiry|Exp\.?)[:\s.]*([A-Z]{3}[.\s/-]?\d{2,4}|\d{1,2}[./-]\d{2,4})', re.I)
mrp_pat = re.compile(r'(?:MRP|Rs\.?|₹)[:\s.]*([\d,.]+)', re.I)

print('OCR text         :', ocr_text[:300])
print('expiry candidates:', date_pat.findall(ocr_text))
print('MRP candidates   :', mrp_pat.findall(ocr_text))

OCR text         : 
expiry candidates: []
MRP candidates   : []


In [5]:
# End-to-end check against the real medicine-mode logic in app/ -- the actual guardrail that
# ships, not a notebook approximation. Only runs when app/ is importable (i.e. locally, or on
# Colab after cloning/uploading the repo); skipped otherwise so the notebook still completes.
import sys
from pathlib import Path

sys.path.insert(0, '..')

try:
    from app.drug_db import DrugDatabase
    from app.modes.medicine import run as run_medicine
except ImportError:
    print('app/ not importable here (expected on a bare Colab runtime) -- skipping.')
    print('Run this cell locally, or upload the drishti/ folder to Colab, to exercise')
    print('the real guardrail. The OCR results above are unaffected.')
else:
    class ReplayOCR:
        """Feeds the OCR text captured above into the real medicine-mode pipeline."""

        def __init__(self, text):
            self._text = text

        def read(self, image_path):
            return self._text

    result = run_medicine(Path('strip.jpg'), ReplayOCR(ocr_text), DrugDatabase.from_file())
    print('ok        :', result.ok)
    print('drug name :', result.drug_name)
    print('expiry    :', result.expiry_raw, '| expired:', result.expired)
    print('MRP       :', result.mrp)
    print('\nspoken    :', result.message_en)

app/ not importable here (expected on a bare Colab runtime) -- skipping.
Run this cell locally, or upload the drishti/ folder to Colab, to exercise
the real guardrail. The OCR results above are unaffected.


## 4. Findings (fill in — goes into the Sem-7 report)

| Check | Result |
|---|---|
| PaddleOCR installs + runs (CPU, no GPU) | |
| Latency per image (CPU) | s |
| Drug name read correctly? | |
| EXP date read correctly? | |
| MRP read correctly? | |
| Devanagari text read correctly? | |
| Guardrail verdict (cell above) | matched / declined |

**If the drug name was read but the guardrail declined it**, that's expected —
`data/drug_names_seed.txt` is a small placeholder list. Add the drug and re-run to confirm
the match path works, then note that sourcing a real CDSCO-derived drug list is an open
task (see `data/README.md` §4).

**If OCR misread the strip**, capture *why* — glare, foil reflection, curved surface, small
print, low contrast. That failure list drives the M3 data-collection protocol in
`docs/data_collection_guide.md` and justifies any preprocessing step you add later.

**Process isolation is itself a finding.** The VLM stack (PyTorch) and the OCR stack
(PaddlePaddle) cannot share a process. On the Android port this stops being a notebook
annoyance and becomes an architectural constraint: the modes in `app/modes/` will need
either a single unified runtime or genuinely separate inference processes. Worth a line in
the report's system-design section.